# 06 - Stage 6.6: Selective prediction demo

Starts the Gradio app with the trained stage 6.2 and 6.3 checkpoints and real stage 6.4 calibration. The app uses CUDA when available. It is a research prototype, not for clinical use.

In [ ]:
import json
import socket
import subprocess
import sys
import torch
from pathlib import Path

root = Path.cwd()
stage66 = root / "results" / "final" / "stage66"
summary_path = stage66 / "stage66_summary.json"
if not summary_path.exists():
    subprocess.run([
        sys.executable, "-m", "src.run_stage66", "--group", "val",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out", str(stage66),
    ], check=True, cwd=root)
print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), indent=2))
port = 7860
with socket.socket() as sock:
    running = sock.connect_ex(("127.0.0.1", port)) == 0
if not running:
    app_process = subprocess.Popen([
        sys.executable, "-m", "src.app",
        "--data-root", str(root / "data" / "Task07_Pancreas"),
        "--checkpoint", str(root / "checkpoints" / "stage62" / "best.pt"),
        "--classifier", str(root / "checkpoints" / "stage63" / "best.pt"),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--port", str(port),
    ], cwd=root)
    print("Started Gradio app; allow a few seconds for model loading.")
else:
    print("Gradio app already responds on this port.")
print("Open http://127.0.0.1:7860 and choose a held-out MSD case.")